In [6]:
# google_trends_enhanced.py
# Enhanced Google Trends Analysis with Multiple Graphics

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Try to import pytrends
try:
    from pytrends.request import TrendReq
    HAS_PYTRENDS = True
except ImportError:
    HAS_PYTRENDS = False
    print("⚠️  pytrends not installed. Install with: pip install pytrends")

def get_data():
    """Fetch live data or generate synthetic data."""
    keywords = ['python', 'data science', 'machine learning']

    if HAS_PYTRENDS:
        try:
            print("📊 Fetching live data from Google Trends...")
            pytrends = TrendReq(hl='en-US', tz=360)
            pytrends.build_payload(kw_list=keywords, timeframe='today 12-m', geo='US')
            df = pytrends.interest_over_time()
            if df is not None and not df.empty:
                print(f"   ✅ Retrieved {len(df)} data points")
                return df, keywords
        except Exception as e:
            print(f"   ⚠️  Error: {e}")

    print("📊 Generating synthetic data...")
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365)
    dates = pd.date_range(start=start_date, end=end_date, freq='D')

    data = {}
    for i, keyword in enumerate(keywords):
        base = 20 + i * 15
        seasonality = 15 * np.sin(np.linspace(0, 2 * np.pi, len(dates)) + i * 1.5)
        trend = np.linspace(0, 5 + i * 3, len(dates))
        noise = np.random.normal(0, 3, len(dates))
        values = np.clip(base + seasonality + trend + noise, 0, 100)
        data[keyword] = values

    df = pd.DataFrame(data, index=dates)
    print(f"   ✅ Generated {len(df)} data points")
    return df, keywords

def chart_1_line_trend(df, keywords):
    """Chart 1: Line chart showing interest over time."""
    plt.figure(figsize=(12, 6))

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    for i, keyword in enumerate(keywords):
        if keyword in df.columns:
            plt.plot(df.index, df[keyword], label=keyword, linewidth=2, color=colors[i])

    plt.title('📈 Search Interest Over Time (12 Months)', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Relative Interest (0-100)')
    plt.legend(loc='upper left')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('google_chart_1_line_trend.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✅ Saved: google_chart_1_line_trend.png")

def chart_2_bar_comparison(df, keywords):
    """Chart 2: Bar chart comparing average interest."""
    plt.figure(figsize=(10, 6))

    avg_values = []
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

    for i, keyword in enumerate(keywords):
        if keyword in df.columns:
            avg_values.append(df[keyword].mean())

    bars = plt.bar(keywords, avg_values, color=colors, edgecolor='white', linewidth=2)

    for bar, value in zip(bars, avg_values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{value:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

    plt.title('📊 Average Interest Comparison', fontsize=14, fontweight='bold')
    plt.ylabel('Average Interest Score')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('google_chart_2_bar_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✅ Saved: google_chart_2_bar_comparison.png")

def chart_3_heatmap(df, keywords):
    """Chart 3: Heatmap showing interest by month and keyword."""
    # Create monthly averages
    df['month'] = df.index.strftime('%Y-%m')
    monthly = df.groupby('month')[keywords].mean()

    # Take last 12 months
    monthly = monthly.tail(12)

    plt.figure(figsize=(12, 8))
    sns.heatmap(monthly.T, annot=True, fmt='.1f', cmap='YlOrRd',
                cbar_kws={'label': 'Interest Score'})
    plt.title('📅 Monthly Interest Heatmap', fontsize=14, fontweight='bold')
    plt.xlabel('Month')
    plt.ylabel('Keyword')
    plt.tight_layout()
    plt.savefig('google_chart_3_heatmap.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✅ Saved: google_chart_3_heatmap.png")

def chart_4_recent_trends(df, keywords):
    """Chart 4: Recent trends (last 30 days) with markers."""
    recent = df.tail(30)

    plt.figure(figsize=(12, 6))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

    for i, keyword in enumerate(keywords):
        if keyword in recent.columns:
            plt.plot(recent.index, recent[keyword], label=keyword, linewidth=2,
                    marker='o', markersize=4, color=colors[i])

    plt.title('📊 Recent Interest Trends (Last 30 Days)', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Relative Interest (0-100)')
    plt.legend(loc='upper left')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('google_chart_4_recent_trends.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✅ Saved: google_chart_4_recent_trends.png")

def main():
    """Main execution function."""
    print("=" * 60)
    print("🔎 GOOGLE TRENDS ANALYSIS - ENHANCED (4 CHARTS)")
    print("=" * 60)

    # Get data
    df, keywords = get_data()

    if df is None or df.empty:
        print("❌ No data available")
        return

    print("\n📈 Generating 4 charts...")
    print("-" * 40)

    # Generate all 4 charts
    chart_1_line_trend(df, keywords)
    chart_2_bar_comparison(df, keywords)
    chart_3_heatmap(df, keywords)
    chart_4_recent_trends(df, keywords)

    print("\n" + "=" * 60)
    print("✅ Complete! 4 charts generated:")
    print("   📊 google_chart_1_line_trend.png")
    print("   📊 google_chart_2_bar_comparison.png")
    print("   📊 google_chart_3_heatmap.png")
    print("   📊 google_chart_4_recent_trends.png")
    print("=" * 60)

if __name__ == "__main__":
    main()

⚠️  pytrends not installed. Install with: pip install pytrends
🔎 GOOGLE TRENDS ANALYSIS - ENHANCED (4 CHARTS)
📊 Generating synthetic data...
   ✅ Generated 366 data points

📈 Generating 4 charts...
----------------------------------------
   ✅ Saved: google_chart_1_line_trend.png
   ✅ Saved: google_chart_2_bar_comparison.png
   ✅ Saved: google_chart_3_heatmap.png
   ✅ Saved: google_chart_4_recent_trends.png

✅ Complete! 4 charts generated:
   📊 google_chart_1_line_trend.png
   📊 google_chart_2_bar_comparison.png
   📊 google_chart_3_heatmap.png
   📊 google_chart_4_recent_trends.png
